In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        os.path.join(dirname, filename)

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
datapath="/kaggle/input/datasets/ejlok1/cremad/AudioWAV"
file_paths=[]
for file in os.listdir(datapath):
    if file.endswith(".wav"):
        file_paths.append(os.path.join(datapath,file))

print("Total files:", len(file_paths))

Total files: 7442


In [3]:
from sklearn.model_selection import train_test_split

train_files, temp_files = train_test_split(
    file_paths, test_size=0.2, random_state=42
)

val_files, test_files = train_test_split(
    temp_files, test_size=0.5, random_state=42
)

In [4]:
def extract_features(file_path, max_len=200, augment=False):
    signal, sr = librosa.load(file_path, sr=None)
    signal, _ = librosa.effects.trim(signal, top_db=20)

    if augment:
        signal = augment_audio(signal, sr)

    mel = librosa.feature.melspectrogram(y=signal, sr=sr, n_mels=128, n_fft=2048, hop_length=512)
    mel = librosa.power_to_db(mel, ref=np.max)

    mfcc = librosa.feature.mfcc(y=signal, sr=sr, n_mfcc=40, n_fft=2048, hop_length=512)
    zcr  = librosa.feature.zero_crossing_rate(signal, hop_length=512)
    rms  = librosa.feature.rms(y=signal, hop_length=512)

    def normalize(f):
        return (f - np.mean(f)) / (np.std(f) + 1e-8)

    mel  = normalize(mel)
    mfcc = normalize(mfcc)
    zcr  = normalize(zcr)
    rms  = normalize(rms)

    combined = np.concatenate([mel, mfcc, zcr, rms], axis=0)  

    if combined.shape[1] < max_len:
        combined = np.pad(combined, ((0, 0), (0, max_len - combined.shape[1])))
    else:
        combined = combined[:, :max_len]

    if augment:
        combined = spec_augment(combined)   

    return combined

In [5]:
def augment_audio(signal,sr):
    if np.random.rand()<0.5:
        signal=librosa.effects.pitch_shift(signal,sr=sr,n_steps=2)
    if np.random.rand()<0.5:
        signal=librosa.effects.time_stretch(signal,rate=0.9)
    if np.random.rand()<0.5:
        noise=0.005*np.random.randn(len(signal))
        signal=signal+noise
    return signal

In [6]:
def spec_augment(mel):
    mel = mel.copy()
    t = np.random.randint(0, 20)
    t0 = np.random.randint(0, mel.shape[1] - t)
    mel[:, t0:t0+t] = 0
    f = np.random.randint(0, 15)
    f0 = np.random.randint(0, mel.shape[0] - f)
    mel[f0:f0+f, :] = 0
    
    return mel

In [7]:
import librosa

emotion_map = {"HAP": 0, "SAD": 1, "ANG": 2, "FEA": 3, "DIS": 4, "NEU": 5}
max_len = 200


X_train, y_train = [], []
for file_path in train_files:
    label = emotion_map[file_path.split("/")[-1].split("_")[2]]
    
    X_train.append(extract_features(file_path, augment=False))
    y_train.append(label)
    
    X_train.append(extract_features(file_path, augment=True))
    y_train.append(label)

X_train = np.array(X_train, dtype=np.float32)
y_train = np.array(y_train, dtype=np.float32)

# VAL
X_val, y_val = [], []
for file_path in val_files:
    X_val.append(extract_features(file_path, augment=False))
    y_val.append(emotion_map[file_path.split("/")[-1].split("_")[2]])

X_val = np.array(X_val, dtype=np.float32)
y_val = np.array(y_val, dtype=np.float32)

# TEST
X_test, y_test = [], []
for file_path in test_files:
    X_test.append(extract_features(file_path, augment=False))
    y_test.append(emotion_map[file_path.split("/")[-1].split("_")[2]])

X_test = np.array(X_test, dtype=np.float32)
y_test = np.array(y_test, dtype=np.float32)



print("X_train shape:", X_train.shape) 
print("X_val shape:",   X_val.shape)
print("X_test shape:",  X_test.shape)

X_train shape: (11906, 170, 200)
X_val shape: (744, 170, 200)
X_test shape: (745, 170, 200)


In [9]:
X_train = np.expand_dims(X_train, -1)
X_val   = np.expand_dims(X_val, -1)
X_test  = np.expand_dims(X_test, -1)

In [10]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D,MaxPooling1D
from tensorflow.keras.layers import LSTM,Dense,Dropout,BatchNormalization,Bidirectional
from tensorflow.keras import regularizers



2026-03-08 04:19:23.293585: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772943563.481449      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772943563.530608      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772943564.006194      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772943564.006222      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772943564.006225      55 computation_placer.cc:177] computation placer alr

# Model with mel spectrogram +conv2d +pooling +flatten +bilstm

In [11]:
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Permute, Reshape

def build_model2(hp):
    l2_value = hp.Choice("l2", [1e-4, 5e-4, 1e-3])
    model = Sequential()

    # Block 1
    model.add(Conv2D(
        hp.Int("filters1", 32, 64, step=32), (3,3),
        activation='relu', padding='same',
        kernel_regularizer=regularizers.l2(l2_value),
        input_shape=(170, 200, 1)
    ))
    model.add(BatchNormalization())
    model.add(MaxPooling2D((2, 2)))

    # Block 2
    model.add(Conv2D(
        hp.Int("filters2", 64, 128, step=32), (3,3),
        activation='relu', padding='same',
        kernel_regularizer=regularizers.l2(l2_value)
    ))
    model.add(BatchNormalization())
    model.add(MaxPooling2D((2, 2)))

    # Block 3
    model.add(Conv2D(
        hp.Int("filters3", 128, 256, step=64), (3,3),
        activation='relu', padding='same',
        kernel_regularizer=regularizers.l2(l2_value)
    ))
    model.add(BatchNormalization())
    model.add(MaxPooling2D((2, 1)))

    # Reshape for LSTM
    model.add(Permute((2, 1, 3)))
    model.add(Reshape((50, -1)))

    # Stacked BiLSTM
    model.add(Bidirectional(
        LSTM(hp.Int("lstm_units1", 64, 128, step=32), return_sequences=True)
    ))
    model.add(Dropout(hp.Float("dropout1", 0.2, 0.4, step=0.1)))

    model.add(Bidirectional(
        LSTM(hp.Int("lstm_units2", 32, 64, step=32), return_sequences=False)
    ))
    model.add(Dropout(hp.Float("dropout2", 0.2, 0.4, step=0.1)))

    model.add(Dense(64, activation='relu'))
    model.add(Dropout(0.3))
    model.add(Dense(6, activation='softmax'))

    model.compile(
        loss='sparse_categorical_crossentropy',
        optimizer=tf.keras.optimizers.Adam(hp.Choice("lr", [1e-3, 3e-4, 1e-4])),
        metrics=['accuracy']
    )
    return model

In [17]:
import shutil


In [14]:
from tensorflow.keras.callbacks import EarlyStopping
import keras_tuner as kt
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=6,
    restore_best_weights=True
)

tuner1=kt.Hyperband(
    build_model2,
    objective='val_accuracy',
    max_epochs=25,
    factor=3,
    directory='tuning_dir2',
    project_name='emotion_model2'
)
tuner1.search(
    X_train,y_train,
    validation_data=(X_val,y_val),
    batch_size=16,
    callbacks=[early_stop]
)

Trial 30 Complete [00h 06m 55s]
val_accuracy: 0.5954301357269287

Best val_accuracy So Far: 0.625
Total elapsed time: 02h 23m 50s


In [16]:
best_model = tuner1.get_best_models(num_models=1)[0]

 best_model.save("/kaggle/working/best_model2.keras")
print("Model saved!")

Model saved!
